<a href="https://colab.research.google.com/github/anshikaguptacs24-del/AI_LAB/blob/main/1BF24CS058_WEEK6_FOL_Unificatin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
def is_variable(x):
    return isinstance(x, str) and x.islower() and x.isalpha()


def apply_substitution(term, subst):
    if isinstance(term, str):
        while term in subst:
            term = subst[term]
        return term

    name, args = term
    return (name, [apply_substitution(a, subst) for a in args])


def occurs(var, term, subst):
    term = apply_substitution(term, subst)

    if term == var:
        return True

    if isinstance(term, tuple):
        return any(occurs(var, a, subst) for a in term[1])

    return False


def unify(x, y, subst):
    x = apply_substitution(x, subst)
    y = apply_substitution(y, subst)

    if x == y:
        return subst

    if is_variable(x):
        if occurs(x, y, subst):
            return None
        subst[x] = y
        return subst

    if is_variable(y):
        if occurs(y, x, subst):
            return None
        subst[y] = x
        return subst

    if isinstance(x, tuple) and isinstance(y, tuple):
        if x[0] != y[0] or len(x[1]) != len(y[1]):
            return None

        for a, b in zip(x[1], y[1]):
            result = unify(a, b, subst)
            if result is None:
                return None

        return subst

    return None


def parse(expr):
    expr = expr.strip()

    if "(" not in expr:
        return expr

    name, rest = expr.split("(", 1)
    rest = rest[:-1]
    args = []
    depth = 0
    start = 0

    for i, ch in enumerate(rest):
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
        elif ch == "," and depth == 0:
            args.append(rest[start:i].strip())
            start = i + 1

    if rest.strip():
        args.append(rest[start:].strip())

    return (name.strip(), [parse(a) for a in args])


def display(term):
    if isinstance(term, str):
        return term

    return term[0] + "(" + ", ".join(
        display(a) for a in term[1]
    ) + ")"


e1 = parse(input("Enter first expression: "))
e2 = parse(input("Enter second expression: "))

result = unify(e1, e2, {})

if result is None:
    print("Unification failed.")
elif not result:
    print("Expressions are identical. No substitution needed.")
else:
    print("Unification successful.")
    print("Most General Unifier:")
    for var, value in result.items():
        print(var, "=", display(value))


Enter first expression: Eats(x,Apple)
Enter second expression: Eats(Riya,y)
Unification successful.
Most General Unifier:
x = Riya
y = Apple
